In [11]:
import subprocess

try:
    subprocess.check_output('nvidia-smi')
    print('Nvidia GPU detected!')
except Exception: # this command not being found can raise quite a few different errors depending on the configuration
    print('No Nvidia GPU in system!')
    

Nvidia GPU detected!


In [17]:
import torch
print(torch.cuda.is_available())
print(torch.backends.cudnn.enabled)

False
True


In [19]:
torch.cuda.device_count()
torch.cuda.current_device()

AssertionError: Torch not compiled with CUDA enabled

In [13]:
import torch
use_cuda = torch.cuda.is_available()

In [15]:
if use_cuda:
    print('__CUDNN VERSION:', torch.backends.cudnn.version())
    print('__Number CUDA Devices:', torch.cuda.device_count())
    print('__CUDA Device Name:',torch.cuda.get_device_name(0))
    print('__CUDA Device Total Memory [GB]:',torch.cuda.get_device_properties(0).total_memory/1e9)

In [16]:
device = torch.device("cuda" if use_cuda else "cpu")
print("Device: ",device)

Device:  cpu


In [3]:
import json
import openai
import os
import pandas as pd
from pprint import pprint
from sklearn.model_selection import train_test_split

client = openai.OpenAI(api_key=os.environ.get(
    "OPENAI_API_KEY"))

In [9]:
system_message = """

You are a helpful assistant. You will be given two sentencs, and you will have to label the semantic relationship between them. 
If the two sentences are semantically related, you will label them as 1. If they are not semantically related, you will label them as 0. 

"""


def create_user_message(row):
    return f"Sentence: {row['sentence1']}\n\nSentence: {row['sentence2']}\n"


def prepare_example_conversation(row):
    return {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": create_user_message(row)},
            {"role": "assistant", "content": row["label"]},
        ]
    }




# predict the rest of the data

In [1]:
job_id = "ftjob-dttfKWZEbLqmzRJXLXtaYEbM"

In [4]:
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model

if fine_tuned_model_id is None:
    raise RuntimeError(
        "Fine-tuned model ID not found. Your job has likely not been completed yet."
    )

print("Fine-tuned model ID:", fine_tuned_model_id)

Fine-tuned model ID: ft:gpt-4o-mini-2024-07-18:personal:recipe-ner:Ay8kQrV0


In [5]:
test_mes = [{'content': '\n'
             '\n'
             'You are a helpful assistant. You will be given two sentencs, and '
             'you will have to label the semantic relationship between them. \n'
             'If the two sentences are semantically related, you will label '
             'them as 1. If they are not semantically related, you will label '
             'them as 0. \n'
             '\n',
             'role': 'system'},
            {'content': 'Sentence: trust in police\n\nSentence: fear of police\n',
             'role': 'user'}]

response = client.chat.completions.create(
    model=fine_tuned_model_id, messages=test_mes, temperature=0, max_tokens=500
)
print(response.choices[0].message.content)

0


In [6]:
# all_variables_pairs_df = pd.read_csv("all_variable_pairs.csv")
all_variables_pairs_df = pd.read_csv("all_definition_pairs.csv")


all_variables_pairs_df.rename(
    columns={"Variable 1": "sentence1", "Variable 2": "sentence2"}, inplace=True)
all_variables_pairs_df.head()

all_variables_pairs_df = all_variables_pairs_df.loc[0:50]

In [10]:
final_results = []

for index, row in all_variables_pairs_df.iterrows():

    test_row = all_variables_pairs_df.iloc[index]

    test_messages = []
    test_messages.append({"role": "system", "content": system_message})
    user_message = create_user_message(test_row)

    test_messages.append({"role": "user", "content": user_message})

    # pprint(test_messages)

    response = client.chat.completions.create(
        model=fine_tuned_model_id, messages=test_messages, temperature=0, max_tokens=500
    )
    # print(response.choices[0].message.content)
    final_results.append(response.choices[0].message.content)

all_variables_pairs_df["predicted_label"] = final_results

In [ ]:
print("shape of the data:", all_variables_pairs_df.shape)


,sentence1,sentence2,predicted_label
0,using alcohol - in reference to misuse,perpetration or victimization of violence by a...,0
1,using alcohol - in reference to misuse,"""Increases"" in mental health indicate a declin...",0
2,using alcohol - in reference to misuse,the occurrence of a (physical mostly) violent ...,0
3,using alcohol - in reference to misuse,perpetration and or victimization of violence ...,0
4,using alcohol - in reference to misuse,reporting an IPV event to the police,0
5,using alcohol - in reference to misuse,discriminatory treatment that Indigenous peopl...,0
6,using alcohol - in reference to misuse,the policies and practices that covertly maint...,0
7,using alcohol - in reference to misuse,police physical violence against indigenous pe...,0
8,using alcohol - in reference to misuse,people getting involved with the police throug...,0
9,using alcohol - in reference to misuse,how helpful or positive interactions with pol...,0
